# Telecom Churn — REST API client notebook

End-to-end lifecycle **strictly through HTTP** (no `ml_core` imports):

1. health check → 2. dataset (synthetic + real upload) → 3. validation & EDA → 4. baseline training →
5. Optuna training → 6. compare & promote → 7. hot reload → 8. inference & sensitivity → 9. monitoring & drift.

Start the API first from `app_ml_rest_api/`:
```bash
uvicorn api.main:app --port 8000
```

In [ ]:
import os, time, json
import numpy as np
import requests
import pandas as pd
import plotly.express as px

API = os.environ.get("API_BASE_URL", "http://localhost:8000")
session = requests.Session()


def call(method: str, path: str, **kw):
    """HTTP helper that raises with the API's error detail."""
    r = session.request(method, f"{API}{path}", timeout=kw.pop("timeout", 120), **kw)
    if not r.ok:
        raise RuntimeError(f"{method} {path} -> {r.status_code}: {r.text[:600]}")
    return r.json()

## 1. Health

In [ ]:
health = call("GET", "/health")
health

## 2. Dataset — synthetic subscribers (and how to add a real file)

In [ ]:
ds = call("POST", "/datasets/synthetic", json={"n_samples": 20000, "target_churn_rate": 0.20, "random_state": 42,
                                                "filename": "notebook_demo.parquet"})
dataset_id = ds["dataset_id"]
print(ds)

# Real (non-synthetic) data, e.g. the Kaggle Telco file:
# with open("WA_Fn-UseC_-Telco-Customer-Churn.csv", "rb") as fh:
#     real = call("POST", "/datasets/upload", files={"file": ("telco.csv", fh)})
#     dataset_id = real["dataset_id"]

## 3. Schema inference, quality gates, EDA

In [ ]:
v = call("POST", "/datasets/validate", json={"dataset_id": dataset_id})
print("target:", v["target"], "| passed:", v["report"]["passed"], "| features:", len(v["feature_columns"]))
print("errors:", v["report"]["errors"], "\nwarnings:", v["report"]["warnings"][:3])
pd.DataFrame(v["schema"]["columns"]).T[["kind", "dtype", "n_unique", "missing_pct"]].head(15)

In [ ]:
eda = call("GET", f"/datasets/{dataset_id}/eda")
print(f"rows={eda['n_rows']:,} positive_rate={eda['positive_rate']:.3f} duplicates={eda['duplicate_rows']}")
pca = call("GET", f"/datasets/{dataset_id}/pca", params={"n_components": 3, "sample_size": 2000})
pts = pd.DataFrame(pca["points"]); pts["churn"] = pca["labels"]
px.scatter(pts, x="PC1", y="PC2", color=pts["churn"].astype(str), opacity=.5,
           title=f"PCA (explained {sum(pca['explained_variance_ratio'][:2]):.1%})")

## 4. Baseline training (background job + polling)

In [ ]:
def wait(job_id: str, every: float = 3.0, timeout: float = 1800.0) -> dict:
    t0 = time.time()
    while time.time() - t0 < timeout:
        job = call("GET", f"/training/jobs/{job_id}")
        print(f"\r{job['status']:<10} {job.get('progress')}", end="")
        if job["status"] in ("succeeded", "failed"):
            print()
            if job["status"] == "failed":
                raise RuntimeError(job["error"])
            return job
        time.sleep(every)
    raise TimeoutError(job_id)


start = call("POST", "/training/start", json={"dataset_id": dataset_id, "config": {
    "model_name": "lightgbm", "mode": "baseline", "sampling": "class_weight", "cv_folds": 5,
    "numeric_imputation": "median", "categorical_imputation": "most_frequent", "decision_threshold": 0.5}})
baseline = wait(start["job_id"])["result"]
print("version:", baseline["model_version"], {k: round(x, 3) for k, x in baseline["test_metrics"].items()})

## 5. Optuna experiment (custom search space, runs without blocking the API)

In [ ]:
start = call("POST", "/training/start", json={"dataset_id": dataset_id, "config": {
    "model_name": "xgboost", "mode": "optuna", "n_trials": 10, "optimize_metric": "pr_auc", "cv_folds": 3,
    "sampling": "smote", "sampling_ratio": 0.6,
    "param_space": {"max_depth": {"type": "int", "low": 3, "high": 7},
                    "learning_rate": {"type": "float", "low": 0.02, "high": 0.2, "log": True}}}})
print("API stays responsive while training:", call("GET", "/health")["status"])
optuna_res = wait(start["job_id"])["result"]
print("version:", optuna_res["model_version"], "best params:", optuna_res["best_params"])

## 6. Compare versions and promote the winner to `@champion`

In [ ]:
versions = pd.DataFrame([{ "version": x["version"], "aliases": x["aliases"], "model": x["tags"].get("model_name"),
                           "mode": x["tags"].get("mode"), "roc_auc": x["metrics"].get("test_roc_auc"),
                           "pr_auc": x["metrics"].get("test_pr_auc")} for x in call("GET", "/models/versions")])
display(versions)
best = int(versions.sort_values("pr_auc", ascending=False).iloc[0]["version"])
print("promoting v", best, call("POST", f"/models/versions/{best}/promote"))
print(call("POST", f"/models/versions/{versions['version'].max()}/alias", json={"alias": "challenger"}))

## 7. Zero-downtime hot reload

In [ ]:
t0 = time.perf_counter()
r = call("POST", "/model/reload")          # defaults to @champion
print(r, f"| wall {1000*(time.perf_counter()-t0):.0f} ms")
print(call("GET", "/health")["model_version"])

## 8. Inference — single, batch, sensitivity

In [ ]:
schema = call("GET", "/model/schema")
features = schema["features"]
record = {f["name"]: f["default"] for f in features}
pred = call("POST", "/predict", json={"features": record, "threshold": schema["threshold"]})
pred

In [ ]:
# Missing fields are imputed (up to MAX_MISSING_FRACTION)
partial = dict(list(record.items())[:-4])
call("POST", "/predict", json={"features": partial})["warnings"]

In [ ]:
# Build 500 plausible subscribers from the model's own feature spec (no filesystem access needed)
rng = np.random.default_rng(42)
def draw(f):
    if f["kind"] == "numeric":
        lo, hi = f["min"], f["max"]
        return rng.uniform(lo, hi, 500) if lo is not None and hi is not None and hi > lo else np.full(500, f["default"])
    return rng.choice(f["categories"], 500)
sample = pd.DataFrame({f["name"]: draw(f) for f in features})
batch = call("POST", "/predict/batch", json={"records": json.loads(sample.to_json(orient="records"))})
print("n:", batch["n"], "predicted churn rate:", round(batch["positive_rate"], 3))
px.histogram(pd.DataFrame(batch["predictions"]), x="churn_probability", nbins=40, title="Batch probabilities")

In [ ]:
num = next(f["name"] for f in features if f["kind"] == "numeric" and f["min"] != f["max"])
cat = next(f["name"] for f in features if f["kind"] == "categorical" and len(f["categories"]) > 1)
s1 = call("POST", "/predict/sensitivity", json={"feature": num, "features": record, "n_points": 30})
s2 = call("POST", "/predict/sensitivity", json={"feature": cat, "features": record})
display(px.line(x=s1["values"], y=s1["probabilities"], labels={"x": num, "y": "P(churn)"}, title=f"Sensitivity: {num}"))
px.bar(x=s2["values"], y=s2["probabilities"], labels={"x": cat, "y": "P(churn)"}, title=f"Sensitivity: {cat}")

## 9. Live telemetry and drift

In [ ]:
call("POST", "/monitoring/reset")
call("POST", "/monitoring/simulate", json={"n": 500, "drift": False, "random_state": 1})
time.sleep(2)   # telemetry is written by a background task *after* the response; give it a moment
summary = call("GET", "/monitoring/summary")
print({k: summary[k] for k in ("window_predictions", "live_positive_rate", "expected_positive_rate", "latency_ms")})
stable = call("GET", "/monitoring/drift")
print("no drift injected ->", stable["overall_status"], "| prediction PSI:", stable["prediction_shift"]["psi"])

In [ ]:
call("POST", "/monitoring/reset")
call("POST", "/monitoring/simulate", json={"n": 500, "drift": True, "random_state": 2})
time.sleep(2)
d = call("GET", "/monitoring/drift")
print("drift injected ->", d["overall_status"])
table = pd.DataFrame([{"feature": k, **v} for k, v in d["features"].items()]).sort_values("psi", ascending=False)
display(table.head(10))
ts = pd.DataFrame(call("GET", "/monitoring/timeseries", params={"bucket_seconds": 5})["buckets"])
px.line(ts, x="t", y=["churn_rate", "rolling_churn_rate"], title="Churn rate over time")